# Briefklar Desk: real Apertus run on Kaggle

Records real Apertus answers for the 23 test letters in `data/bench_small` and scores them.

**Before you run:**
1. Settings: **Accelerator → GPU T4 x2**, **Internet → on**.
2. Accept the conditions on https://huggingface.co/swiss-ai/Apertus-8B-Instruct-2509 with the same Hugging Face account as your token.
3. **Add-ons → Secrets**: `HF_TOKEN` (a Read token), switched on for this notebook.

Then **Run All**. Cell 1 stops everything if the model cannot be served. The whole run takes about an hour.
At the end, download `apertus_results.zip` from the **Output** panel.

Model used: Apertus 8B Instruct (September 2025 release), fp16, text only (OCR text, no image), served with vLLM with JSON-constrained output. Apertus 1.5 could not be loaded with the public libraries.

In [ ]:
# 1. Serve Apertus with vLLM on both GPUs. Stops here if that fails.
import os, subprocess, time, json, urllib.request
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
MODEL = "swiss-ai/Apertus-8B-Instruct-2509"

!pip install -q uv
!uv venv -q /kaggle/vllm --python 3.12
!uv pip install -q --python /kaggle/vllm/bin/python vllm

server = subprocess.Popen(
    ["/kaggle/vllm/bin/vllm", "serve", MODEL, "--served-model-name", MODEL, "--dtype", "half",
     "--tensor-parallel-size", "2", "--max-model-len", "8192", "--gpu-memory-utilization", "0.90", "--port", "8000",
     "--enforce-eager", "--max-num-seqs", "16"],  # T4: no torch.compile / CUDA graphs
    stdout=open("/kaggle/working/vllm.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())

ready = False
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5); ready = True; break
    except Exception:
        if server.poll() is not None: break
        time.sleep(10)
if not ready:
    log = open("/kaggle/working/vllm.log").read().splitlines()
    print("\n".join([l for l in log if "Error" in l or "ERROR" in l][-25:] or log[-40:]))
    raise RuntimeError("FAIL: vLLM could not serve the model. Copy the log above into a new chat.")

request = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", method="POST",
    headers={"Content-Type": "application/json"},
    data=json.dumps({"model": MODEL, "temperature": 0, "max_tokens": 60,
                     "messages": [{"role": "user", "content": "Was ist ein Zahlungsbefehl? Ein Satz."}]}).encode())
print(json.load(urllib.request.urlopen(request, timeout=300))["choices"][0]["message"]["content"])
print("SUCCESS: the model is served")

In [ ]:
# 2. Tesseract for OCR, and the project code
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-deu tesseract-ocr-fra tesseract-ocr-ita > /dev/null
!rm -rf /kaggle/working/briefklar-desk && git clone -q --depth 1 https://github.com/bhavya-jpg/briefklar-desk.git /kaggle/working/briefklar-desk
%cd /kaggle/working/briefklar-desk/track_2b
!uv sync -q --frozen
!tesseract --list-langs

In [ ]:
# 3. Record real answers for the 23 test letters, then score them (about an hour)
os.environ.update(
    PYTHONPATH="src:.", LLM_BASE_URL="http://127.0.0.1:8000/v1", LLM_NAME=MODEL, LLM_API_KEY="none",
    BRIEFKLAR_VISION="0",       # this release reads text only: Pass 1 gets the OCR text
    BRIEFKLAR_TOOLS="0",        # no tool parser for this model in vLLM: the orchestrator runs every tool itself
    BRIEFKLAR_THINKING="0", BRIEFKLAR_GUIDED_JSON="1", BRIEFKLAR_LLM_TIMEOUT="600",
    BRIEFKLAR_REPLAY_DIR="/kaggle/working/briefklar-desk/track_2b/data/replay_apertus")
!uv run python -m bench.fixtures.record --live --only bench 2>&1 | grep -E "FAILED|recorded|responses in"
!uv run python -m bench.eval.run --mode replay --results-dir bench/eval/results/apertus_kaggle 2>&1 | grep -E "^(bench_small|model outputs|metric|action|deadline|letter_type|field_f1|red_flag|triage|thread|open_total|reply)" 

In [ ]:
# 4. Package the results: download apertus_results.zip from the Output panel
!cp /kaggle/working/vllm.log bench/eval/results/apertus_kaggle/vllm.log
!zip -qr /kaggle/working/apertus_results.zip data/replay_apertus bench/eval/results/apertus_kaggle
print(json.load(open("data/replay_apertus/MANIFEST.json")))
print("Done. Download apertus_results.zip from the Output panel on the right.")
server.terminate()